In [10]:
!nvidia-smi

Sun Mar 15 16:54:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             33W /   70W |    2183MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [33]:
%cd /content

!rm -rf hoankiem-air-quality-forecasting

!git clone https://github.com/HoangHumg1210/hoankiem-air-quality-forecasting.git

%cd hoankiem-air-quality-forecasting

!ls

/content
Cloning into 'hoankiem-air-quality-forecasting'...
remote: Enumerating objects: 80, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 80 (delta 32), reused 67 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (80/80), 8.59 MiB | 15.41 MiB/s, done.
Resolving deltas: 100% (32/32), done.
/content/hoankiem-air-quality-forecasting
data  notebooks  requirements.txt  src


In [ ]:
# import os, sys

# PROJECT_ROOT = "/content/hoankiem-air-quality"  # thư mục chứa folder src/
# assert os.path.isdir(PROJECT_ROOT), f"Không thấy thư mục: {PROJECT_ROOT}"

# if PROJECT_ROOT not in sys.path:
#     sys.path.insert(0, PROJECT_ROOT)  # ưu tiên path này

In [34]:
import math
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.ensemble import RandomForestRegressor

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# from fix_data_leakage import scale_data_without_leakage, create_sequences
from sklearn.preprocessing import StandardScaler

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline




In [35]:
from src.data_utils import (
    set_seed,
    load_and_clean_data,
    add_time_features,
    add_target_features,
    split_data,
    transform_target,
    preprocess_features,
    create_sequences
)

ImportError: cannot import name 'set_seed' from 'src.data_utils' (/content/hoankiem-air-quality-forecasting/src/data_utils.py)

In [ ]:
TARGET = 'PM25'
LOOKBACK = 336
HORIZON = 8
USE_LOG_TARGET = True
DATA_PATH = 'data/processed/data2225_done.csv'


In [ ]:
def prepare_sequences_for_experiment(
    data_path: str,
    target: str = "PM25",
    transform: str = "log",  # "log" / "sqrt" / "none"
    lookback: int = 336,
    horizon: int = 72,
):
    """Chuẩn bị X_seq, y_seq cho 1 cấu hình thí nghiệm.

    Hàm này lặp lại đúng pipeline trong `src.data_utils` nhưng cho phép
    thay đổi transform target và tham số LOOKBACK / HORIZON.
    """
    df = load_and_clean_data(data_path)
    df = add_time_features(df)
    df = add_target_features(df, target=target)

    train_df, val_df, test_df = split_data(df)

    # 1) Transform target theo config
    y_train, y_val, y_test, scaler_y, inverse_target_transform = transform_target(
        train_df,
        val_df,
        test_df,
        target=target,
        transform=transform,
    )

    # 2) Preprocess features
    X_train, X_val, X_test, preprocess = preprocess_features(
        train_df,
        val_df,
        test_df,
        target=target,
    )

    # 3) Tạo sequence
    X_train_seq, y_train_seq = create_sequences(X_train, y_train, lookback=lookback, horizon=horizon)
    X_val_seq, y_val_seq = create_sequences(X_val, y_val, lookback=lookback, horizon=horizon)
    X_test_seq, y_test_seq = create_sequences(X_test, y_test, lookback=lookback, horizon=horizon)

    return {
        "df": df,
        "train_df": train_df,
        "val_df": val_df,
        "test_df": test_df,
        "X_train_seq": X_train_seq,
        "y_train_seq": y_train_seq,
        "X_val_seq": X_val_seq,
        "y_val_seq": y_val_seq,
        "X_test_seq": X_test_seq,
        "y_test_seq": y_test_seq,
        "scaler_y": scaler_y,
        "inverse_target_transform": inverse_target_transform,
        "preprocess": preprocess,
        "lookback": lookback,
        "horizon": horizon,
        "transform": transform,
    }


def build_cnn_gru_model(
    n_features: int,
    lookback: int,
    horizon: int,
    arch: str = "heavy"
):
    """Xây dựng kiến trúc CNN-GRU cho multi-step forecasting."""
    if arch == "heavy":
        model = Sequential([
            Conv1D(
                filters=64,
                kernel_size=3,
                activation="relu",
                padding="same",
                input_shape=(lookback, n_features)
            ),
            Conv1D(filters=64, kernel_size=3, activation="relu", padding="same"),
            MaxPooling1D(pool_size=2),
            Dropout(0.2),

            GRU(256, return_sequences=True),
            Dropout(0.2),
            GRU(128, return_sequences=True),
            Dropout(0.2),
            GRU(64, return_sequences=False),
            Dropout(0.2),

            Dense(32, activation="relu"),
            Dense(horizon),
        ])

    elif arch == "light":
        model = Sequential([
            GRU(128, return_sequences=True, input_shape=(lookback, n_features)),
            Dropout(0.1),
            GRU(64, return_sequences=False),
            Dropout(0.1),
            Dense(32, activation="relu"),
            Dense(horizon),
        ])
    else:
        raise ValueError(f"Unknown arch type: {arch}")

    return model

In [ ]:
prepare_sequences_for_experiment(
    data_path=DATA_PATH,
    target=TARGET,
    transform="none",
    lookback=LOOKBACK,
    horizon=HORIZON,
)


In [ ]:
experiment_configs = [
    {
        "name": "A_no_log_MSE_h72_l336_cnn_gru_heavy",
        "transform": "none",
        "lookback": 336,
        "horizon": 72,   # 72 giờ = 3 ngày
        "loss": "mse",
        "arch": "heavy",
    }
]

results_summary = []


def inverse_scale_2d(arr_2d, scaler):
    original_shape = arr_2d.shape
    return scaler.inverse_transform(arr_2d.reshape(-1, 1)).reshape(original_shape)


def aggregate_72h_to_3days(arr_2d):
    """
    arr_2d: shape (N, 72)
    return: shape (N, 3)
    """
    day1 = arr_2d[:, 0:24].mean(axis=1)
    day2 = arr_2d[:, 24:48].mean(axis=1)
    day3 = arr_2d[:, 48:72].mean(axis=1)
    return np.stack([day1, day2, day3], axis=1)


for cfg in experiment_configs:
    print("\n" + "=" * 80)
    print(f"Running experiment: {cfg['name']}")
    print("Config:", cfg)

    artifacts = prepare_sequences_for_experiment(
        data_path=DATA_PATH,
        target=TARGET,
        transform=cfg["transform"],
        lookback=cfg["lookback"],
        horizon=cfg["horizon"],
    )

    X_train_seq = artifacts["X_train_seq"]
    y_train_seq = artifacts["y_train_seq"]
    X_val_seq = artifacts["X_val_seq"]
    y_val_seq = artifacts["y_val_seq"]
    X_test_seq = artifacts["X_test_seq"]
    y_test_seq = artifacts["y_test_seq"]
    scaler_y = artifacts["scaler_y"]
    inverse_target_transform = artifacts["inverse_target_transform"]

    print("X_train_seq shape:", X_train_seq.shape)
    print("y_train_seq shape:", y_train_seq.shape)

    if len(X_train_seq) == 0 or len(X_val_seq) == 0:
        print("WARNING: empty sequences for this config, skip.")
        continue

    n_features = X_train_seq.shape[2]
    lookback = cfg["lookback"]
    horizon = cfg["horizon"]

    model = build_cnn_gru_model(
        n_features=n_features,
        lookback=lookback,
        horizon=horizon,
        arch=cfg["arch"],
    )

    loss_name = cfg["loss"].lower()
    if loss_name == "mse":
        loss_fn = tf.keras.losses.MeanSquaredError()
    elif loss_name == "mae":
        loss_fn = tf.keras.losses.MeanAbsoluteError()
    else:
        raise ValueError(f"Unsupported loss in this cell: {loss_name}")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss=loss_fn,
        metrics=["mae"],
    )

    # 1 weight / sample
    thr = np.quantile(y_train_seq, 0.90)
    w_train = np.where((y_train_seq > thr).any(axis=1), 4.0, 1.0).astype(np.float32)

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=1,
    )

    lr_scheduler = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1,
    )

    history = model.fit(
        X_train_seq,
        y_train_seq,
        sample_weight=w_train,
        epochs=100,
        batch_size=64,
        validation_data=(X_val_seq, y_val_seq),
        callbacks=[early_stop, lr_scheduler],
        verbose=1,
    )

    # Predict 72 giờ
    y_pred_scaled = model.predict(X_test_seq)

    # inverse scaling
    y_pred_t = inverse_scale_2d(y_pred_scaled, scaler_y)
    y_test_t_inv = inverse_scale_2d(y_test_seq, scaler_y)

    # inverse target transform
    y_pred_raw = inverse_target_transform(y_pred_t)
    y_test_raw_inv = inverse_target_transform(y_test_t_inv)

    # gộp 72 giờ -> 3 ngày
    y_pred_3d = aggregate_72h_to_3days(y_pred_raw)
    y_true_3d = aggregate_72h_to_3days(y_test_raw_inv)

    # Metric từng ngày
    mae_day1 = mean_absolute_error(y_true_3d[:, 0], y_pred_3d[:, 0])
    mae_day2 = mean_absolute_error(y_true_3d[:, 1], y_pred_3d[:, 1])
    mae_day3 = mean_absolute_error(y_true_3d[:, 2], y_pred_3d[:, 2])

    rmse_day1 = math.sqrt(mean_squared_error(y_true_3d[:, 0], y_pred_3d[:, 0]))
    rmse_day2 = math.sqrt(mean_squared_error(y_true_3d[:, 1], y_pred_3d[:, 1]))
    rmse_day3 = math.sqrt(mean_squared_error(y_true_3d[:, 2], y_pred_3d[:, 2]))

    # Metric gộp cả 3 ngày
    mae_all = mean_absolute_error(y_true_3d.reshape(-1), y_pred_3d.reshape(-1))
    rmse_all = math.sqrt(mean_squared_error(y_true_3d.reshape(-1), y_pred_3d.reshape(-1)))

    print(f"{cfg['name']} - MAE D+1:  {mae_day1:.4f}")
    print(f"{cfg['name']} - MAE D+2:  {mae_day2:.4f}")
    print(f"{cfg['name']} - MAE D+3:  {mae_day3:.4f}")
    print(f"{cfg['name']} - MAE All:  {mae_all:.4f}")

    print(f"{cfg['name']} - RMSE D+1: {rmse_day1:.4f}")
    print(f"{cfg['name']} - RMSE D+2: {rmse_day2:.4f}")
    print(f"{cfg['name']} - RMSE D+3: {rmse_day3:.4f}")
    print(f"{cfg['name']} - RMSE All: {rmse_all:.4f}")

    results_summary.append(
        {
            "name": cfg["name"],
            "transform": cfg["transform"],
            "lookback": cfg["lookback"],
            "horizon": cfg["horizon"],
            "loss": cfg["loss"],
            "arch": cfg["arch"],
            "mae_day1": float(mae_day1),
            "mae_day2": float(mae_day2),
            "mae_day3": float(mae_day3),
            "mae_all": float(mae_all),
            "rmse_day1": float(rmse_day1),
            "rmse_day2": float(rmse_day2),
            "rmse_day3": float(rmse_day3),
            "rmse_all": float(rmse_all),
        }
    )

    # Plot D+1
    plt.figure(figsize=(14, 4))
    n_plot = min(120, len(y_true_3d))
    plt.plot(y_true_3d[:n_plot, 0], label="True D+1", alpha=0.8)
    plt.plot(y_pred_3d[:n_plot, 0], label="Pred D+1", alpha=0.8)
    plt.title(f"{cfg['name']} - PM2.5 ngày mai (D+1)")
    plt.xlabel("Test samples")
    plt.ylabel("PM2.5 daily mean")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    # Plot D+2
    plt.figure(figsize=(14, 4))
    plt.plot(y_true_3d[:n_plot, 1], label="True D+2", alpha=0.8)
    plt.plot(y_pred_3d[:n_plot, 1], label="Pred D+2", alpha=0.8)
    plt.title(f"{cfg['name']} - PM2.5 ngày kia (D+2)")
    plt.xlabel("Test samples")
    plt.ylabel("PM2.5 daily mean")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    # Plot D+3
    plt.figure(figsize=(14, 4))
    plt.plot(y_true_3d[:n_plot, 2], label="True D+3", alpha=0.8)
    plt.plot(y_pred_3d[:n_plot, 2], label="Pred D+3", alpha=0.8)
    plt.title(f"{cfg['name']} - PM2.5 ngày thứ 3 (D+3)")
    plt.xlabel("Test samples")
    plt.ylabel("PM2.5 daily mean")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


print("\n=== Tóm tắt kết quả các cấu hình ===")
for r in results_summary:
    print(
        f"{r['name']}: transform={r['transform']}, loss={r['loss']}, "
        f"lookback={r['lookback']}, horizon={r['horizon']} | "
        f"MAE(D+1/D+2/D+3/All)=({r['mae_day1']:.4f}, {r['mae_day2']:.4f}, {r['mae_day3']:.4f}, {r['mae_all']:.4f}) | "
        f"RMSE(D+1/D+2/D+3/All)=({r['rmse_day1']:.4f}, {r['rmse_day2']:.4f}, {r['rmse_day3']:.4f}, {r['rmse_all']:.4f})"
    )

In [ ]:
# === Experiments: target transform + loss + horizon/lookback + architecture ===

experiment_configs = [
    {
        "name": "A_no_log_MSE_h8_l336_cnn_gru_heavy",
        "transform": "none",
        "lookback": 168,
        "horizon": 3,
        "loss": "mse",
        "arch": "heavy",
    },
    {
        "name": "B_log_MAE_h8_l336_cnn_gru_heavy",
        "transform": "log",
        "lookback": 336,
        "horizon": 8,
        "loss": "mae",
        "arch": "heavy",
    },
    {
        "name": "C_no_log_MSE_h1_l168_cnn_gru_light",
        "transform": "none",
        "lookback": 168,
        "horizon": 1,
        "loss": "mse",
        "arch": "light",
    },
]

results_summary = []

for cfg in experiment_configs:
    print("\n" + "=" * 80)
    print(f"Running experiment: {cfg['name']}")
    print("Config:", cfg)

    # 1) Chuẩn bị dữ liệu & sequence cho cấu hình này
    artifacts = prepare_sequences_for_experiment(
        data_path=DATA_PATH,
        target=TARGET,
        transform=cfg["transform"],
        lookback=cfg["lookback"],
        horizon=cfg["horizon"],
    )

    X_train_seq = artifacts["X_train_seq"]
    y_train_seq = artifacts["y_train_seq"]
    X_val_seq = artifacts["X_val_seq"]
    y_val_seq = artifacts["y_val_seq"]
    X_test_seq = artifacts["X_test_seq"]
    y_test_seq = artifacts["y_test_seq"]
    scaler_y = artifacts["scaler_y"]
    inverse_target_transform = artifacts["inverse_target_transform"]

    if len(X_train_seq) == 0 or len(X_val_seq) == 0:
        print("WARNING: empty sequences for this config, skip.")
        continue

    n_features = X_train_seq.shape[2]
    lookback = cfg["lookback"]

    # 2) Xây dựng model
    model = build_cnn_gru_model(n_features=n_features, lookback=lookback, arch=cfg["arch"])

    # 3) Chọn loss function
    loss_name = cfg["loss"].lower()
    if loss_name == "mse":
        loss_fn = tf.keras.losses.MeanSquaredError()
    elif loss_name == "mae":
        loss_fn = tf.keras.losses.MeanAbsoluteError()
    else:
        raise ValueError(f"Unsupported loss in this cell: {loss_name}")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss=loss_fn,
        metrics=["mae"],
    )

    # 4) Tăng trọng số cho các điểm y_train_seq cao (trong không gian đã transform + scale)
    thr = np.quantile(y_train_seq, 0.90)
    w_train = np.where(y_train_seq > thr, 4.0, 1.0).astype(np.float32)

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=1,
    )
    lr_scheduler = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1,
    )

    history = model.fit(
        X_train_seq,
        y_train_seq,
        sample_weight=w_train,
        epochs=100,
        batch_size=64,
        validation_data=(X_val_seq, y_val_seq),
        callbacks=[early_stop, lr_scheduler],
        verbose=1,
    )

    # 5) Dự báo trên test và tính MAE / RMSE trong đơn vị PM2.5 gốc
    y_pred_scaled = model.predict(X_test_seq)
    y_pred_t = scaler_y.inverse_transform(y_pred_scaled)
    y_test_t_inv = scaler_y.inverse_transform(y_test_seq)

    y_pred_raw = inverse_target_transform(y_pred_t)
    y_test_raw_inv = inverse_target_transform(y_test_t_inv)

    mae_val = mean_absolute_error(y_test_raw_inv, y_pred_raw)
    rmse_val = math.sqrt(mean_squared_error(y_test_raw_inv, y_pred_raw))

    print(f"{cfg['name']} - MAE:  {mae_val:.4f}")
    print(f"{cfg['name']} - RMSE: {rmse_val:.4f}")

    # Lưu lại để so sánh sau
    results_summary.append(
        {
            "name": cfg["name"],
            "transform": cfg["transform"],
            "lookback": cfg["lookback"],
            "horizon": cfg["horizon"],
            "loss": cfg["loss"],
            "arch": cfg["arch"],
            "mae": float(mae_val),
            "rmse": float(rmse_val),
        }
    )

    # 6) Vẽ nhanh đồ thị so sánh y_true vs y_pred trên 1 đoạn test
    plt.figure(figsize=(14, 4))
    n_plot = min(500, len(y_test_raw_inv))
    plt.plot(y_test_raw_inv[:n_plot], label="True", alpha=0.8)
    plt.plot(y_pred_raw[:n_plot], label="Pred", alpha=0.8)
    plt.title(f"{cfg['name']} - PM2.5 thực tế vs dự báo (đoạn đầu test)")
    plt.xlabel("Time steps (test subset)")
    plt.ylabel("PM2.5")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


print("\n=== Tóm tắt kết quả các cấu hình ===")
for r in results_summary:
    print(
        f"{r['name']}: transform={r['transform']}, loss={r['loss']}, "
        f"lookback={r['lookback']}, horizon={r['horizon']} | "
        f"MAE={r['mae']:.4f}, RMSE={r['rmse']:.4f}"
    )


In [ ]:
set_seed()
df = load_and_clean_data(DATA_PATH)
df.head()

In [ ]:
# 2 feature engineering
df = add_time_features(df)
df = add_target_features(df)

# 3 split time-series
train_df, val_df, test_df = split_data(df)


In [ ]:

# 4 transform target
y_train, y_val, y_test, scaler_y, inverse_target_transform = transform_target(
    train_df, val_df, test_df
)

# 5 preprocess features
X_train, X_val, X_test, preprocess = preprocess_features(
    train_df, val_df, test_df
)

# 6 create sequences
X_train_seq, y_train_seq = create_sequences(X_train, y_train)
X_val_seq, y_val_seq = create_sequences(X_val, y_val)
X_test_seq, y_test_seq = create_sequences(X_test, y_test)


In [ ]:
# Raw target
y_train_raw = train_df[[TARGET]].values
y_val_raw   = val_df[[TARGET]].values
y_test_raw  = test_df[[TARGET]].values

def forward(y):
    y = np.asarray(y, dtype=np.float64)
    return np.log1p(np.clip(y, 0, None))

y_train_t = forward(y_train_raw)
y_val_t   = forward(y_val_raw)
y_test_t  = forward(y_test_raw)

scaler_y = StandardScaler()
y_train_s = scaler_y.fit_transform(y_train_t)
y_val_s   = scaler_y.transform(y_val_t)
y_test_s  = scaler_y.transform(y_test_t)

fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=False)

# 1) Raw
axes[0].plot(train_df.index, y_train_raw.ravel(), label='Train', alpha=0.8)
axes[0].plot(val_df.index,   y_val_raw.ravel(),   label='Val',   alpha=0.8)
axes[0].plot(test_df.index,  y_test_raw.ravel(),  label='Test',  alpha=0.8)
axes[0].set_title('Bước 1: PM2.5 gốc (Raw)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('PM2.5 (µg/m³)')
axes[0].legend(); axes[0].grid(alpha=0.3)

# 2) Log1p
axes[1].plot(train_df.index, y_train_t.ravel(), label='Train', alpha=0.8)
axes[1].plot(val_df.index,   y_val_t.ravel(),   label='Val',   alpha=0.8)
axes[1].plot(test_df.index,  y_test_t.ravel(),  label='Test',  alpha=0.8)
axes[1].set_title('Bước 2: Sau log1p transform', fontsize=13, fontweight='bold')
axes[1].set_ylabel('log1p(PM2.5)')
axes[1].legend(); axes[1].grid(alpha=0.3)

# 3) Scaled
axes[2].plot(train_df.index, y_train_s.ravel(), label='Train', alpha=0.8)
axes[2].plot(val_df.index,   y_val_s.ravel(),   label='Val',   alpha=0.8)
axes[2].plot(test_df.index,  y_test_s.ravel(),  label='Test',  alpha=0.8)
axes[2].axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Mean (train)')
axes[2].set_title('Bước 3: Sau StandardScaler (fit trên Train)', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Scaled value')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Histogram
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(y_train_raw.ravel(), bins=50, alpha=0.7, color='#4e79a7', edgecolor='white')
axes[0].set_title('Phân phối PM2.5 gốc (Train)'); axes[0].set_xlabel('PM2.5')

axes[1].hist(y_train_t.ravel(), bins=50, alpha=0.7, color='#f28e2b', edgecolor='white')
axes[1].set_title('Phân phối sau log1p (Train)'); axes[1].set_xlabel('log1p(PM2.5)')

axes[2].hist(y_train_s.ravel(), bins=50, alpha=0.7, color='#e15759', edgecolor='white')
axes[2].set_title('Phân phối sau StandardScaler (Train)'); axes[2].set_xlabel('Scaled value')

plt.tight_layout()
plt.show()

print("=== Thống kê qua từng bước ===")
for name, data in [("Raw", y_train_raw), ("Log1p", y_train_t), ("Scaled", y_train_s)]:
    d = data.ravel()
    print(f"{name:8s} | Mean: {d.mean():8.3f} | Std: {d.std():7.3f} | "
          f"Min: {d.min():8.3f} | Max: {d.max():7.3f} | Skew: {pd.Series(d).skew():6.3f}")


In [ ]:
print("Kích thước tập train:", train_df.shape)
print("Kích thước tập validation:", val_df.shape)
print("Kích thước tập test:", test_df.shape)

In [ ]:
n_features = X_train_seq.shape[2]  
model_cnn_gru = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', 
           padding='same', input_shape=(LOOKBACK, n_features)),
    Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
    MaxPooling1D(pool_size=2),
    Dropout(0.2),
    GRU(256, return_sequences=True),
    Dropout(0.2),
    GRU(128,  return_sequences=True),
    Dropout(0.2),
    GRU(64, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1) # Output layer cho dự đoán PM2.5
])

model_cnn_gru.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.Huber(delta=1.0),
    metrics=['mae']
)
model_cnn_gru.summary()



In [ ]:
thr = np.quantile(y_train_seq, 0.90)
w_train = np.where(y_train_seq > thr, 4.0, 1.0).astype(np.float32)
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)


early_stop_cnn_gru = EarlyStopping(
    monitor='val_loss', 
    patience=10, 
    restore_best_weights=True,
    verbose=1
)
history_cnn_gru = model_cnn_gru.fit(
    X_train_seq, y_train_seq,
    sample_weight=w_train,
    epochs=100,
    batch_size=64,
    validation_data=(X_val_seq, y_val_seq),
    callbacks=[early_stop_cnn_gru, lr_scheduler],
    verbose=1
)

In [ ]:
n_features = X_train_seq.shape[2]

model_gru = Sequential([
    tf.keras.Input(shape=(LOOKBACK, n_features)),
    GRU(256, return_sequences=True),
    Dropout(0.2),
    GRU(128, return_sequences=True),
    Dropout(0.2),
    GRU(64, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

# model_gru.compile(optimizer='adam', loss='mae', metrics=['mae'])
model_gru.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.Huber(delta=1.0),   
    metrics=['mae']
)
model_gru.summary()


In [ ]:

y_pred_gru_scaled = model_cnn_gru.predict(X_test_seq)

y_pred_t = scaler_y.inverse_transform(y_pred_gru_scaled)
y_test_t_inv = scaler_y.inverse_transform(y_test_seq)

y_pred_raw = inverse_target_transform(y_pred_t)
y_test_raw_inv = inverse_target_transform(y_test_t_inv)


mae_cnn_gru  = mean_absolute_error(y_test_raw_inv, y_pred_raw)
rmse_cnn_gru = math.sqrt(mean_squared_error(y_test_raw_inv, y_pred_raw))

print(f"CNN-GRU - MAE:  {mae_cnn_gru:.4f}")
print(f"CNN-GRU - RMSE: {rmse_cnn_gru:.4f}")

In [ ]:
N = 300

plt.figure(figsize=(16, 5))
plt.plot(y_test_raw_inv[:N], label='Thực tế', alpha=0.8)
plt.plot(y_pred_raw[:N],     label='CNN-GRU', alpha=0.8)


plt.legend()
plt.title(f'PM2.5 - Thực tế vs CNN-GRU {N} mẫu đầu')
plt.ylabel('PM2.5')
plt.show()